# 共享变量

In [1]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

sparkBuilder = SparkSession.builder \
    .appName("rddDemo") \
    .master("local[2]") \
    .config("spark.log.level", "WARN") \
    .getOrCreate()

sc = sparkBuilder.sparkContext

----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 62374)
Traceback (most recent call last):
  File "D:\Environment\anaconda3\envs\spark35\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "D:\Environment\anaconda3\envs\spark35\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "D:\Environment\anaconda3\envs\spark35\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "D:\Environment\anaconda3\envs\spark35\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "D:\Environment\anaconda3\envs\spark35\lib\site-packages\pyspark\accumulators.py", line 295, in handle
    poll(accum_updates)
  File "D:\Environment\anaconda3\envs\spark35\lib\site-packages\pyspark\accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "

## 累计器

In [2]:
counter = sc.accumulator(0)
rdd = sc.parallelize(range(10))

def increment(x):
    # global counter
    # counter += x
    counter.add(x) # 调用 Accumulator 方法，只改内部值，不会修改值类型

rdd.foreach(increment)
print("Counter value", counter.value)

Counter value 45


## 广播变量

定义在Executor端，Executor端的所有task都可以读取（**只读**）这个变量

In [3]:
b = sc.broadcast([1,2,3,4,5,6])
print(b.value)
b1 = sc.parallelize([0,0]).flatMap(lambda x: b.value).collect()
print(b1)
b.unpersist()

[1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6]


## RDD的依赖关系



窄依赖：一个父RDD的Partition只能被一个子RDD的Partition使用，一对一
宽依赖：一个父RDD的Partition会被多个子RDD的Partition使用，一对多

### DAG（有向无环图）

通过宽依赖来划分stage，窄依赖在一个stage内

### RDD的持久化

未避免多次重复计算统一RDD算子，将RDD缓存在磁盘或者内存中，已达到持久化

持久化实例

In [4]:
rdd = sc.parallelize(['b', 'a', 'c'])

In [5]:
rdd.persist()
level = rdd.getStorageLevel()
print(level)

Memory Serialized 1x Replicated


In [8]:
rdd2 = sc.parallelize(['b', 'a', 'c'])
rdd2.persist(StorageLevel.DISK_ONLY_3)
level = rdd2.getStorageLevel()

# rdd.saveAsTextFile(path=)
print(level)

ConnectionRefusedError: [WinError 10061] 由于目标计算机积极拒绝，无法连接。